# 07 · LLM-as-judge: grading outputs that have no gold label

Free-text outputs (justifications, summaries, topic names) can't be scored with accuracy. A judge model can grade them, **but only after the judge itself is validated**. This notebook:

1. **Rubric judge** (`ARC_JUDGE`): are the sentiment-arc labels supported, and are the justifications faithful to the chat? Typed output, reasoning first.
2. **Meta-evaluation with synthetic corruptions**: we break outputs on purpose (justifications swapped in from a chat of another domain; final label flipped) and measure whether the judge catches them, and how often it falsely flags untouched outputs.
3. **Pairwise adjudication** (`FINAL_SENTIMENT_PAIRWISE`) of the cases where the model disagreed with the synthetic gold label (notebook 04 found they cluster at the neutral boundary). Each pair is asked in **both orders**; only order-consistent verdicts count. A control set (gold vs a deliberately wrong label) checks that the pairwise judge works.
4. **Human spot check**: export a blind sample to CSV, label it yourself, and measure judge–human agreement (κ).

Known biases to keep in mind: position bias (handled by order swapping), self-preference (use a judge from a different family than the predictor when possible), verbosity bias (the rubric says not to reward length).

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os
import random

import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import cohen_kappa_score

from genaianalysis.run import run_task
from genaianalysis.schema import load_jsonl
from genaianalysis.tasks.judge import (
    ARC_JUDGE,
    FINAL_SENTIMENT_PAIRWISE,
    with_candidate,
    with_options,
)
from genaianalysis.tasks.sentiment import SENTIMENT_ARC
from genaianalysis.utils.paths import data_processed_dir, project_dir

load_dotenv(project_dir(".env"))
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")
logging.basicConfig(level=logging.ERROR)
pd.set_option("display.max_colwidth", 200)

PREDICTOR = "google:gemini-flash-lite-latest"  # same as notebook 04, so predictions are cache hits
# A judge from another family avoids self-preference; falls back to the predictor's family.
JUDGE = "anthropic:claude-sonnet-5" if os.getenv("ANTHROPIC_API_KEY") else "google:gemini-flash-lite-latest"
RPM = 12
FIELDS = ["initial_sentiment", "initial_sentiment_justification", "final_sentiment", "final_sentiment_justification"]
print("judge:", JUDGE, "| same family as predictor:", JUDGE.split(":")[0] == PREDICTOR.split(":")[0])

## 0 · Predictions to judge

In [ ]:
chats = load_jsonl(data_processed_dir("synthetic_es_45.jsonl"))
by_id = {c.id: c for c in chats}
arcs = await run_task(SENTIMENT_ARC, chats, PREDICTOR, rpm=RPM)
arcs = arcs[arcs["error"].isna()].reset_index(drop=True)
for f in ["initial_sentiment", "final_sentiment"]:
    arcs[f"gold_{f}"] = arcs["id"].map(lambda i, f=f: by_id[i].labels[f])
print(len(arcs), "predictions")

## 1–2 · Rubric judge on original and corrupted outputs

In [ ]:
rng = random.Random(0)
FLIP = {"positive": "negative", "negative": "positive", "neutral": "negative"}
cases = []
for _, row in arcs.iterrows():
    conv, cand = by_id[row["id"]], row[FIELDS].to_dict()
    donor = arcs[arcs["id"].map(lambda i: by_id[i].meta["domain"]) != conv.meta["domain"]].sample(1, random_state=rng.randrange(10**6)).iloc[0]
    variants = {
        "original": cand,
        "swapped_justification": cand | {f: donor[f] for f in FIELDS if f.endswith("justification")},
        "flipped_final": cand | {"final_sentiment": FLIP[cand["final_sentiment"]]},
    }
    cases += [with_candidate(conv, v, kind) for kind, v in variants.items()]

judged = await run_task(ARC_JUDGE, cases, JUDGE, rpm=RPM)
judged["kind"] = judged["id"].str.split(":").str[-1]
judged["chat_id"] = judged["id"].str.split(":").str[0]
print(judged["error"].notna().sum(), "judge errors")

What a good judge should do: **original** → mostly supported and faithful; **swapped_justification** → `justifications_faithful = False`; **flipped_final** → `final_label_supported = False`.

In [ ]:
ok = judged[judged["error"].isna()]
summary = ok.groupby("kind")[["initial_label_supported", "final_label_supported", "justifications_faithful"]].mean()
summary["mean_score"] = ok.groupby("kind")["score"].mean()
summary["n"] = ok.groupby("kind").size()
summary.round(2)

In [ ]:
detect = pd.Series({
    "swapped justification caught (faithful=False)": 1 - summary.loc["swapped_justification", "justifications_faithful"],
    "flipped final label caught (final_supported=False)": 1 - summary.loc["flipped_final", "final_label_supported"],
    "false alarm on originals (faithful=False)": 1 - summary.loc["original", "justifications_faithful"],
}).round(2)
detect

On the originals, compare the judge's `final_label_supported` with whether the prediction matched the synthetic gold. Disagreement here is either judge error or gold-label noise, which section 3 separates.

In [ ]:
pred = arcs[["id", "final_sentiment", "gold_final_sentiment"]].rename(columns={"id": "chat_id", "final_sentiment": "pred_final"})
orig = ok[ok["kind"] == "original"].merge(pred, on="chat_id")
pd.crosstab(orig["pred_final"] == orig["gold_final_sentiment"], orig["final_label_supported"],
            rownames=["prediction == gold"], colnames=["judge: final label supported"])

## 3 · Pairwise adjudication: model vs synthetic gold
For each disagreement ask: which label fits the chat better, in both orders. **Control**: on agreements, pit gold against a clearly wrong label; the judge should pick gold nearly always.

In [ ]:
QUESTION = {
    "initial_sentiment": "What is the customer's sentiment at the beginning of the conversation?",
    "final_sentiment": "What is the customer's sentiment at the end of the conversation?",
}
pairs, meta = [], []
for f, q in QUESTION.items():
    for _, row in arcs.iterrows():
        gold, pred = row[f"gold_{f}"], row[f]
        if gold != pred:
            kind, other = "dispute", pred
        elif rng.random() < 0.35:
            kind, other = "control", FLIP[gold]
        else:
            continue
        for order, (a, b) in {"gold_first": (gold, other), "other_first": (other, gold)}.items():
            pairs.append(with_options(by_id[row["id"]], q, a, b, f"{f}:{kind}:{order}"))
            meta.append({"id": pairs[-1].id, "field": f, "kind": kind, "order": order, "gold": gold, "other": other})

pw = (await run_task(FINAL_SENTIMENT_PAIRWISE, pairs, JUDGE, rpm=RPM)).merge(pd.DataFrame(meta), on="id")
pw["picked_gold"] = ((pw["order"] == "gold_first") & (pw["choice"] == "A")) | ((pw["order"] == "other_first") & (pw["choice"] == "B"))
pw["chat_id"] = pw["id"].str.split(":").str[0]
v = pw[pw["error"].isna()].pivot_table(index=["chat_id", "field", "kind"], columns="order", values="picked_gold", aggfunc="first").dropna()
v["consistent"] = v["gold_first"] == v["other_first"]
v["verdict"] = v["gold_first"].map({True: "gold", False: "model"}).where(v["consistent"], "inconsistent")
v.groupby(["kind", "field"])["verdict"].value_counts().unstack(fill_value=0)

In [ ]:
print("position consistency:", round(v["consistent"].mean(), 2))
print("share of A picks (0.5 = no position bias):", round((pw["choice"] == "A").mean(), 2))

Disputes where the judge sided with the model, i.e. probable gold-label noise:

In [ ]:
sided = v[(v["verdict"] == "model")].reset_index()
sided = sided.merge(pw[["chat_id", "field", "gold", "other", "reasoning"]].drop_duplicates(["chat_id", "field"]), on=["chat_id", "field"])
sided[["chat_id", "field", "gold", "other", "reasoning"]].head(8)

## 4 · Human spot check
Export a blind sample (no judge verdicts), fill in `human_final_label_supported` and `human_faithful` with 1/0, save, and re-run the next cell.

In [ ]:
SPOT = data_processed_dir("judge_spotcheck.csv")
if not SPOT.exists():
    sample = ok[ok["kind"].isin(["original", "flipped_final"])].sample(20, random_state=0)
    blind = pd.DataFrame({
        "id": sample["id"],
        "chat": sample["chat_id"].map(lambda i: by_id[i].to_text()),
        "analysis": sample["id"].map({c.id: c.messages[-1].text for c in cases}),
        "human_final_label_supported": "", "human_faithful": "",
    })
    blind.to_csv(SPOT, index=False)
    print("wrote", SPOT)

In [ ]:
human = pd.read_csv(SPOT)
filled = human.dropna(subset=["human_final_label_supported", "human_faithful"])
if len(filled):
    m = filled.merge(ok, on="id")
    print("n =", len(m))
    print("κ final_label_supported:", round(cohen_kappa_score(m["human_final_label_supported"].astype(bool), m["final_label_supported"]), 2))
    print("κ faithful:", round(cohen_kappa_score(m["human_faithful"].astype(bool), m["justifications_faithful"]), 2))
else:
    print("spot-check file not labeled yet:", SPOT)

## Cost
Judging is not free: compare against the cost of the predictions it grades.

In [ ]:
pd.Series({
    "prediction $/1k": arcs["cost_usd"].mean() * 1000,
    "rubric judge $/1k": judged["cost_usd"].mean() * 1000,
    "pairwise judge $/1k": pw["cost_usd"].mean() * 1000,
}).round(3)

## Findings
Record in `docs/research.md`: detection rates, false alarms, position consistency, and how many disputes the judge attributes to gold-label noise.